# Student Performance & Academic Insights
**Mini Project | Exploratory Data Analysis | Python, Pandas, NumPy, Matplotlib & Seaborn**

## 1. Aim
Analyze student academic performance, attendance, study hours, and assignment completion to identify patterns that may help students improve their results.

## 2. Problem Statement
Marks alone do not explain why students perform differently. This project explores how attendance, study time, assignment completion, and subject-wise marks relate to overall academic performance.

## 3. Objectives
- Create and inspect a structured student-performance dataset.
- Clean missing values and check duplicate records.
- Calculate total marks, average marks, and performance categories.
- Visualize subject performance, attendance, study hours, and correlations.
- Summarize useful observations from the analysis.

## 4. Tools Used
Python, Pandas, NumPy, Matplotlib, Seaborn, Jupyter Notebook.

## 5. Dataset Description
For easy execution, this notebook creates a **sample dataset** with 120 fictional students. It includes attendance percentage, weekly study hours, assignment completion, and marks in Mathematics, Programming, and Computer Networks. The data is synthetic and should not be treated as real student records.

## 6. Importing required libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 2)

print("Libraries imported successfully.")

## 7. Creating the sample dataset

In [ ]:
# Fixed seed makes the sample data reproducible
rng = np.random.default_rng(42)
n = 120

attendance = np.clip(rng.normal(82, 10, n), 45, 100).round(1)
study_hours = np.clip(rng.normal(10, 4, n), 1, 25).round(1)
assignment_completion = np.clip(rng.normal(78, 15, n), 20, 100).round(1)

# Marks are illustrative and influenced slightly by attendance and study habits
base = 25 + 0.28 * attendance + 1.25 * study_hours + 0.12 * assignment_completion
maths = np.clip(base + rng.normal(0, 10, n) - 25, 0, 100).round(1)
programming = np.clip(base + rng.normal(0, 11, n) - 22, 0, 100).round(1)
networks = np.clip(base + rng.normal(0, 12, n) - 24, 0, 100).round(1)

df = pd.DataFrame({
    "Student_ID": [f"STU{i:03d}" for i in range(1, n + 1)],
    "Attendance_Percent": attendance,
    "Study_Hours_Per_Week": study_hours,
    "Assignment_Completion_Percent": assignment_completion,
    "Mathematics_Marks": maths,
    "Programming_Marks": programming,
    "Computer_Networks_Marks": networks
})

# Add a few missing values to demonstrate data cleaning
for col in ["Attendance_Percent", "Programming_Marks", "Computer_Networks_Marks"]:
    idx = rng.choice(df.index, size=3, replace=False)
    df.loc[idx, col] = np.nan

print("Dataset created:", df.shape)
display(df.head())

## 8. Exploring the dataset

In [ ]:
print("Number of rows and columns:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())
print("\nData types and non-null counts:")
df.info()

In [ ]:
display(df.describe().T)

## 9. Missing values and duplicate records

In [ ]:
print("Missing values in each column:")
display(df.isna().sum().rename("Missing_Count").to_frame())

print("Duplicate rows:", df.duplicated().sum())
print("Duplicate student IDs:", df["Student_ID"].duplicated().sum())

In [ ]:
# Fill missing numeric values with each column's median
clean_df = df.copy()
numeric_cols = clean_df.select_dtypes(include="number").columns
clean_df[numeric_cols] = clean_df[numeric_cols].fillna(clean_df[numeric_cols].median())
clean_df = clean_df.drop_duplicates(subset="Student_ID")

print("Missing values after cleaning:", int(clean_df.isna().sum().sum()))
print("Cleaned dataset shape:", clean_df.shape)

## 10. Feature engineering and performance categories

In [ ]:
subject_cols = ["Mathematics_Marks", "Programming_Marks", "Computer_Networks_Marks"]
clean_df["Average_Marks"] = clean_df[subject_cols].mean(axis=1).round(2)

clean_df["Performance_Category"] = pd.cut(
    clean_df["Average_Marks"],
    bins=[-0.1, 39.99, 59.99, 74.99, 100],
    labels=["Needs Improvement", "Average", "Good", "Excellent"]
)

display(clean_df[["Student_ID", *subject_cols, "Average_Marks", "Performance_Category"]].head(10))
print("\nStudents in each performance category:")
display(clean_df["Performance_Category"].value_counts().rename_axis("Category").to_frame("Students"))

## 11. Subject-wise average marks

In [ ]:
subject_averages = clean_df[subject_cols].mean().sort_values(ascending=False)
display(subject_averages.rename("Average_Marks").to_frame())

plt.figure(figsize=(9, 5))
sns.barplot(x=subject_averages.index, y=subject_averages.values)
plt.title("Average Marks by Subject")
plt.xlabel("Subject")
plt.ylabel("Average Marks")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 12. Attendance and average marks

In [ ]:
plt.figure(figsize=(8, 5))
sns.regplot(data=clean_df, x="Attendance_Percent", y="Average_Marks",
            scatter_kws={"alpha": 0.65}, line_kws={"color": "red"})
plt.title("Attendance Percentage vs Average Marks")
plt.xlabel("Attendance (%)")
plt.ylabel("Average Marks")
plt.tight_layout()
plt.show()

print("Correlation between attendance and average marks:",
      round(clean_df["Attendance_Percent"].corr(clean_df["Average_Marks"]), 3))

## 13. Weekly study hours and average marks

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=clean_df, x="Study_Hours_Per_Week", y="Average_Marks",
                hue="Performance_Category", alpha=0.8)
plt.title("Weekly Study Hours vs Average Marks")
plt.xlabel("Study Hours per Week")
plt.ylabel("Average Marks")
plt.tight_layout()
plt.show()

## 14. Distribution of student performance

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(clean_df["Average_Marks"], bins=12, kde=True)
plt.title("Distribution of Average Marks")
plt.xlabel("Average Marks")
plt.ylabel("Number of Students")
plt.tight_layout()
plt.show()

## 15. Correlation heatmap

In [ ]:
corr_cols = [
    "Attendance_Percent", "Study_Hours_Per_Week",
    "Assignment_Completion_Percent", *subject_cols, "Average_Marks"
]
plt.figure(figsize=(10, 7))
sns.heatmap(clean_df[corr_cols].corr(), annot=True, cmap="coolwarm", fmt=".2f", center=0)
plt.title("Correlation Between Study Habits and Academic Results")
plt.tight_layout()
plt.show()

## 16. Assignment completion and performance

In [ ]:
clean_df["Assignment_Group"] = pd.cut(
    clean_df["Assignment_Completion_Percent"],
    bins=[0, 50, 75, 100],
    labels=["0–50%", "51–75%", "76–100%"],
    include_lowest=True
)
assignment_summary = clean_df.groupby("Assignment_Group", observed=False)["Average_Marks"].mean()
display(assignment_summary.rename("Average_Marks").to_frame())

plt.figure(figsize=(7, 5))
sns.barplot(x=assignment_summary.index, y=assignment_summary.values)
plt.title("Average Marks by Assignment Completion")
plt.xlabel("Assignment Completion")
plt.ylabel("Average Marks")
plt.tight_layout()
plt.show()

## 17. Key findings generated from this dataset

In [ ]:
best_subject = clean_df[subject_cols].mean().idxmax().replace("_Marks", "").replace("_", " ")
weakest_subject = clean_df[subject_cols].mean().idxmin().replace("_Marks", "").replace("_", " ")
attendance_corr = clean_df["Attendance_Percent"].corr(clean_df["Average_Marks"])
study_corr = clean_df["Study_Hours_Per_Week"].corr(clean_df["Average_Marks"])
top_category = clean_df["Performance_Category"].value_counts().idxmax()

print(f"1. Highest average subject: {best_subject}.")
print(f"2. Lowest average subject: {weakest_subject}.")
print(f"3. Most common performance category: {top_category}.")
print(f"4. Attendance-to-average-marks correlation: {attendance_corr:.2f}.")
print(f"5. Study-hours-to-average-marks correlation: {study_corr:.2f}.")
print("\nNote: These findings describe only the synthetic dataset, not real students.")

## 18. Conclusion
This mini-project demonstrates how Python can be used to inspect, clean, analyze, and visualize academic data. It calculates average marks and performance categories, then explores relationships between academic results and attendance, study hours, and assignment completion.

The visualizations can help identify areas for further investigation. However, correlation does not prove causation, and this sample dataset is fictional. Real academic decisions should use verified data and consider additional factors.

## 19. Future Enhancements
- Replace the generated sample data with an authorized real dataset.
- Add semester-wise performance comparisons.
- Build an interactive dashboard using Streamlit or Power BI.
- Compare results across departments or student cohorts while protecting privacy.
- If suitable real data is available, evaluate a prediction model using proper train/test splitting and fairness checks.

**How to run:** Open this notebook in Jupyter Notebook or Google Colab and choose **Run All**. No external CSV file or API key is required.